In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
import pysam

In [2]:

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU dla M1 pro")
elif torch.backends.cuda.is_available():
    device = torch.device("cuda")
    print("GPU dla Nvidaia")
else:
    device = torch.device("cpu")
    print("idzie na cpu")


GPU dla M1 pro


In [3]:
class dna_killer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder =nn.Sequential(
            nn.Conv1d(in_channels=4, out_channels=32,kernel_size=3,padding=1),
            nn.BatchNorm1d(32), # skaluje do 0 1
            nn.ReLU(),          # ujemne wartosci ustawia na 0
            nn.MaxPool1d(2),     # Zmniejszamy długość o połowę (np. 100 -> 50)


            nn.Conv1d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
            # drop (wył.cznie neurnow)            
        )

        self.decoder=nn.Sequential(
            nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm1d(32),
            nn.ReLU(), 
            
            nn.ConvTranspose1d(32, 4, kernel_size=2, stride=2),  
            

            nn.Sigmoid()
        )
    def forward(self, x):
        
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
        

     